# 01 — Sanity checks

**Purpose:** prove the data pipeline and the model are correct *before* 22 runs depend on
them. Every cell below is a check with a right answer; none of them is exploratory.

Prerequisite: `uv run python scripts/get_data.py --all`.

Build ladder rung 6.1 / 6.2 in `docs/orientation.md`.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

## 1. Read the raw format

Read the first 1,000 rows of `data/test.h5` with `pandas.read_hdf(..., start=, stop=)`.
Confirm: 806 columns, `E_i / PX_i / PY_i / PZ_i` for i in 0..199, plus `is_signal_new`.

**Expected:** signal fraction near 0.5, constituents pT-ordered, padding exactly zero.

**This is a gate, not a warm-up.** The evaluation set is the *first 200k rows* of this
file. That is only legitimate if the file is shuffled. If the signal fraction of the
first 200k is outside [0.45, 0.55], switch to the seed-pinned random slice that
`preregistration.md` §2 already commits to, and log the deviation.


In [ ]:
import pandas as pd
df = pd.read_hdf(ROOT / "data" / "test.h5", key="table", start=0, stop=1000)
print(df.shape)
print([c for c in df.columns[:8]], "...", [c for c in df.columns[-8:]])
print("signal fraction:", df["is_signal_new"].mean())

## 2. Constituent multiplicity, and what the 64-cap costs

Histogram the number of constituents with E > 0. Then compute, per jet, the fraction of
total jet pT carried by the leading 64.

**This settles deviation D-003.** If the median captured fraction is > 99%, say so in
`docs/deviations.md` with the number and downgrade the entry to benign.

In [ ]:
# your turn: n_constituents histogram, and pT-fraction captured by the leading 64

## 3. Plot one jet

Scatter (Δη, Δφ) with marker area ∝ pT, one top jet and one QCD jet side by side.

**Expected:** the top jet shows a hard core plus additional prongs (three-prong
substructure, though smeared); QCD is typically one dominant blob. If both look
identical, your Δη/Δφ are probably not relative to the jet axis.

In [ ]:
# your turn: two-panel jet display

## 4. The one plot that validates the whole feature pipeline

Reconstruct the jet four-vector as the sum of constituent four-vectors and histogram its
invariant mass, split by label.

**Expected:** signal peaks near the top mass (~175 GeV); background falls steeply. If the
signal peak is missing or in the wrong place, something in your four-vector handling is
wrong and *no* later result is meaningful.

In [ ]:
# your turn: m = sqrt(E^2 - px^2 - py^2 - pz^2) of the summed constituents, by label

## 5. Feature sanity

Build the 7 features. Check: no NaN/inf (log of a padded zero is the classic bug), ΔR
below ~0.8 for essentially all constituents (anti-kT R=0.8), log pT_rel ≤ 0 always.

In [ ]:
# your turn: from jetscaling.data import build_features, read_four_vectors

## 6. Model checks

1. **Permutation invariance:** shuffle constituents within each jet — max |Δlogit| < 1e-5.
2. **Padding invariance:** pad the same jets to a different length — logits unchanged.
3. **Parameter counts:** all four rungs within ~25% of the targets in `configs/model_sizes.yaml`.

Checks 1 and 2 are how you catch the mask bug described in `orientation.md` §5.4, which
otherwise produces a *better-looking* tagger and a scaling law about nothing.

In [ ]:
# your turn

## 7. Overfit 1,000 jets

Train the `tiny` model on 1,000 jets until training loss is near zero.

**Expected:** loss → ~0 within a few hundred steps. If it cannot memorise 1,000 jets, the
bug is in the model, the mask or the optimiser — find it now, not after the sweep.
Note the wall time per step: multiply by the largest cell's step count for a first
estimate of whether the `large` row fits the two-day budget (fallback: drop it).

In [ ]:
# your turn